# Breast Cancer Detection - EfficientNet-B4 V12 Pro

## CBIS-DDSM Mammography Classification Pipeline (Enhanced)

**Architecture:** EfficientNet-B4 with Transfer Learning

**Version:** V12 Pro - Enhanced Preprocessing & A100-Optimized Training

**Dataset:** CBIS-DDSM Region of Interest (ROI) Cropped Images

---

### Key Enhancements (V12 Pro)

1. **Enhanced Preprocessing Pipeline**
   - Multi-stage CLAHE with adaptive parameters
   - Proper normalization for EfficientNet [-1, 1]
   - Optional histogram equalization pre-CLAHE
   
2. **Advanced Data Augmentation**
   - MixUp augmentation for better generalization
   - CutMix for spatial regularization
   - Random erasing for occlusion robustness
   - Rotation with medical imaging constraints
   
3. **A100-Optimized Configuration**
   - Batch size 32 with gradient accumulation fallback
   - 3-model ensemble with diversity
   - Mixed precision FP16 for 2x throughput
   
4. **Improved Training Strategy**
   - Conservative unfreezing (Stage 2 focus from V13 learnings)
   - Cosine annealing with warm restarts
   - Best checkpoint selection across ALL stages
   - Automatic threshold calibration

5. **Previous Best Results**
   - V12: 0.7955 AUC (Model 2, Stage 2)
   - V13: 0.7870 AUC (Model 2, Stage 2)
   - **Target: >0.82 AUC**

---

### Pipeline Structure

1. Environment Configuration
2. Enhanced Preprocessing Pipeline
3. Advanced Augmentation
4. Model Architecture with Spatial Dropout
5. Optimized 3-Stage Training
6. Comprehensive Evaluation

## 1. Environment Setup

Configure runtime environment, mount storage, and initialize GPU memory management.

In [ ]:
import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNTIME_ENV = "colab"
except ImportError:
    RUNTIME_ENV = "local"

if RUNTIME_ENV == "colab":
    import subprocess
    subprocess.run(['pip', 'install', 'pydicom', '-q'], capture_output=True)

import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

import cv2
import pydicom
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

physical_gpus = tf.config.list_physical_devices('GPU')
GPU_AVAILABLE = len(physical_gpus) > 0

# Auto-detect GPU memory in GB
def get_gpu_memory_gb():
    """Auto-detect GPU memory in GB. Returns conservative estimate for Colab."""
    if not GPU_AVAILABLE:
        return 0
    try:
        gpu_details = tf.config.experimental.get_device_details(physical_gpus[0])
        gpu_name = gpu_details.get('device_name', '').lower()

        # Known GPU memory sizes (conservative estimates)
        if 'h100' in gpu_name:
            return 80  # H100 has 80GB
        elif 'a100' in gpu_name:
            return 40  # A100 has 40GB (or 80GB for A100-80GB)
        elif 'l4' in gpu_name:
            return 24  # L4 has 24GB
        elif 't4' in gpu_name:
            return 15  # T4 has 16GB but reserve some
        elif 'v100' in gpu_name:
            return 16  # V100 has 16GB or 32GB
        elif 'p100' in gpu_name:
            return 16
        elif 'k80' in gpu_name:
            return 12
        else:
            # Try to detect from name patterns
            if '80gb' in gpu_name or '80g' in gpu_name:
                return 80
            elif '40gb' in gpu_name or '40g' in gpu_name:
                return 40
            # Default conservative for unknown GPUs
            return 15
    except Exception:
        # Fallback - check if we can get memory info directly
        try:
            # Try nvidia-smi approach
            import subprocess
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                                   capture_output=True, text=True)
            if result.returncode == 0:
                mem_mb = int(result.stdout.strip().split('\n')[0])
                return mem_mb // 1024  # Convert MB to GB
        except Exception:
            pass
        return 15  # Fallback for Colab T4

GPU_MEMORY_GB = get_gpu_memory_gb()

if GPU_AVAILABLE:
    for gpu in physical_gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    GPU_NAME = tf.config.experimental.get_device_details(physical_gpus[0]).get('device_name', 'Unknown')
else:
    GPU_NAME = "None"

# Helper to print memory usage
def print_memory_usage(stage=""):
    """Print current memory usage for debugging."""
    if RUNTIME_ENV == "colab":
        try:
            import psutil
            mem = psutil.virtual_memory()
            print(f"[{stage}] RAM: {mem.used/1e9:.1f}GB / {mem.total/1e9:.1f}GB ({mem.percent}%)")
        except ImportError:
            pass
    if GPU_AVAILABLE:
        try:
            gpu_mem = tf.config.experimental.get_memory_info('GPU:0')
            if gpu_mem:
                print(f"[{stage}] GPU: {gpu_mem.get('current', 0)/1e9:.2f}GB")
        except Exception:
            pass

print("-" * 60)
print("ENVIRONMENT CONFIGURATION")
print("-" * 60)
print(f"Runtime:        {RUNTIME_ENV.upper()}")
print(f"TensorFlow:     {tf.__version__}")
print(f"NumPy:          {np.__version__}")
print(f"GPU Available:  {GPU_AVAILABLE}")
if GPU_AVAILABLE:
    print(f"GPU Name:       {GPU_NAME}")
    print(f"GPU Memory:     {GPU_MEMORY_GB} GB (detected)")
print("-" * 60)

## 2. Configuration Parameters

Define hyperparameters optimized for EfficientNet-B4 architecture. Input size set to 380x380 to leverage EfficientNet-B4 native resolution for improved feature extraction.

In [ ]:
class ExperimentConfig:
    """
    V12 Pro Configuration - Enhanced for A100 with learnings from V12/V13.
    
    Key improvements:
    - Stronger regularization (L2=2e-4, dropout=0.45, spatial_dropout=0.15)
    - Conservative unfreezing (350 frozen in Stage 2, 150 in Stage 3)
    - Stage 2 focus with extended epochs
    - Best checkpoint selection across all stages
    """
    
    def __init__(self, gpu_memory_gb=15):
        self.gpu_memory_gb = gpu_memory_gb
        self.runtime_env = RUNTIME_ENV
        self.version = "V12_Pro"
        
        # Data Paths
        if self.runtime_env == "colab":
            self.base_path = Path('/content/drive/MyDrive/CBIS-DDSM-data')
            self.dicom_path = self.base_path / 'CBIS-DDSM'
            self.csv_path = self.base_path / 'csv'
            self.cache_path = self.base_path / 'preprocessed_cache_v12pro'
            self.checkpoint_path = self.base_path / 'checkpoints_v12pro'
            self.results_path = self.base_path / 'results_v12pro'
        else:
            self.base_path = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
            self.dicom_path = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
            self.csv_path = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
            self.cache_path = self.base_path / 'preprocessed_cache_v12pro'
            self.checkpoint_path = self.base_path / 'checkpoints_v12pro'
            self.results_path = self.base_path / 'results_v12pro'
        
        # CSV paths
        self.calc_train_csv = self.csv_path / 'calc_case_description_train_set.csv'
        self.calc_test_csv = self.csv_path / 'calc_case_description_test_set.csv'
        self.mass_train_csv = self.csv_path / 'mass_case_description_train_set.csv'
        self.mass_test_csv = self.csv_path / 'mass_case_description_test_set.csv'
        
        # EfficientNet-B4 native resolution
        self.image_size = (380, 380)
        self.image_channels = 3
        
        # ============================================================
        # ENHANCED PREPROCESSING CONFIGURATION
        # ============================================================
        self.use_clahe = True
        self.clahe_clip_limit = 2.5  # Slightly higher for ROI images
        self.clahe_tile_size = (8, 8)
        self.use_histogram_eq_pre = False  # Optional pre-CLAHE equalization
        self.use_bilateral_filter = True   # Noise reduction while preserving edges
        self.bilateral_d = 5
        self.bilateral_sigma_color = 50
        self.bilateral_sigma_space = 50
        
        # ============================================================
        # GPU-ADAPTIVE CONFIGURATION
        # ============================================================
        if gpu_memory_gb >= 40:
            # A100 (40GB) - FULL CONFIGURATION
            self.batch_size = 32
            self.batch_size_finetune = 16
            self.batch_size_inference = 64
            self.dense_units = 1024
            self.ensemble_size = 3
            self.use_mixed_precision = True
            self.tta_augmentations = 8
            self.shuffle_buffer = 2000
            self.gradient_accumulation_steps = 1
        elif gpu_memory_gb >= 20:
            # L4 (24GB) or V100-32GB
            self.batch_size = 16
            self.batch_size_finetune = 8
            self.batch_size_inference = 32
            self.dense_units = 1024
            self.ensemble_size = 3
            self.use_mixed_precision = True
            self.tta_augmentations = 8
            self.shuffle_buffer = 1000
            self.gradient_accumulation_steps = 2
        elif gpu_memory_gb >= 15:
            # T4 (15GB)
            self.batch_size = 8
            self.batch_size_finetune = 4
            self.batch_size_inference = 16
            self.dense_units = 512
            self.ensemble_size = 1
            self.use_mixed_precision = True
            self.tta_augmentations = 4
            self.shuffle_buffer = 500
            self.gradient_accumulation_steps = 4
        else:
            # Low memory
            self.batch_size = 4
            self.batch_size_finetune = 2
            self.batch_size_inference = 8
            self.dense_units = 256
            self.ensemble_size = 1
            self.use_mixed_precision = False
            self.tta_augmentations = 4
            self.shuffle_buffer = 500
            self.gradient_accumulation_steps = 8
        
        self.ensemble_seeds = [42, 123, 456][:self.ensemble_size]
        
        # ============================================================
        # OPTIMIZED TRAINING SCHEDULE (Learning from V12/V13)
        # ============================================================
        # Stage 1: Train head - moderate epochs
        self.stage1_epochs = 25
        self.stage1_lr = 3e-4  # Reduced from 5e-4 (V13 showed better stability)
        
        # Stage 2: Fine-tune top layers - FOCUS STAGE (V12/V13 showed best results here)
        self.stage2_epochs = 60  # Extended - this is where best results come from
        self.stage2_lr = 8e-5   # Lower LR for stability
        self.stage2_frozen_layers = 350  # Conservative (EfficientNet-B4 ~474 layers)
        
        # Stage 3: Full fine-tune - Reduced (often overfits)
        self.stage3_epochs = 20  # Reduced from 80 (V13 showed Stage 3 often degrades)
        self.stage3_lr = 1e-5   # Very low LR
        self.stage3_frozen_layers = 150  # More conservative than before
        
        # Learning rate scheduling
        self.use_lr_warmup = True
        self.warmup_epochs = 5
        self.warmup_start_lr = 1e-7
        self.use_cosine_annealing = True
        self.cosine_min_lr = 1e-7
        
        # Data split
        self.train_split = 0.70
        self.val_split = 0.15
        self.test_split = 0.15
        
        # ============================================================
        # ENHANCED REGULARIZATION (V13 learnings)
        # ============================================================
        self.l1_regularization = 0.0
        self.l2_regularization = 2e-4  # Between V12 (1e-4) and V13 (3e-4)
        self.dropout_rate = 0.45       # Between V12 (0.4) and V13 (0.5)
        self.label_smoothing = 0.12    # Slightly increased
        self.gradient_clip_norm = 1.0
        
        # Spatial dropout (new for V12 Pro)
        self.use_spatial_dropout = True
        self.spatial_dropout_rate = 0.15
        
        # ============================================================
        # ADVANCED AUGMENTATION
        # ============================================================
        self.use_mixup = True
        self.mixup_alpha = 0.2
        self.use_cutmix = True
        self.cutmix_alpha = 0.2
        self.mixup_prob = 0.3  # Probability of applying mixup per batch
        self.cutmix_prob = 0.3  # Probability of applying cutmix per batch
        
        self.use_random_erasing = True
        self.random_erasing_prob = 0.25
        self.random_erasing_scale = (0.02, 0.15)
        
        # Rotation augmentation
        self.use_rotation = True
        self.rotation_range = 15  # degrees - conservative for medical images
        
        # ============================================================
        # CLASS WEIGHTING (adjusted based on V13 threshold analysis)
        # ============================================================
        self.use_class_weight = True
        self.malignant_weight_multiplier = 2.0  # Reduced from 2.5 to reduce malignant bias
        self.class_weights = None
        
        # Focal Loss (adjusted)
        self.use_focal_loss = True
        self.focal_gamma = 2.0
        self.focal_alpha = 0.65  # Reduced from 0.7 to reduce malignant bias
        
        # ============================================================
        # IMPROVED CALLBACKS
        # ============================================================
        self.early_stop_patience = 20  # Between V12 (25) and V13 (15)
        self.lr_reduce_patience = 8
        self.lr_reduce_factor = 0.5
        self.min_epochs_stage1 = 15
        self.min_epochs_stage2 = 30  # Extended minimum for Stage 2
        self.min_epochs_stage3 = 10  # Reduced for Stage 3
        
        # Best checkpoint selection
        self.select_best_across_stages = True  # NEW: Select best from any stage
        
        # TTA
        self.use_tta = True
        
        # Performance
        self.prefetch_buffer = tf.data.AUTOTUNE
        self.num_parallel_calls = tf.data.AUTOTUNE
        
        self._create_directories()
    
    def _create_directories(self):
        for path in [self.cache_path, self.checkpoint_path, self.results_path]:
            path.mkdir(parents=True, exist_ok=True)
    
    def display(self):
        total_epochs = (self.stage1_epochs + self.stage2_epochs + self.stage3_epochs)
        total_training_epochs = total_epochs * self.ensemble_size
        
        print("=" * 65)
        print("V12 PRO - ENHANCED CONFIGURATION")
        print("=" * 65)
        print(f"GPU Memory:           {self.gpu_memory_gb} GB")
        print(f"Mixed Precision:      {self.use_mixed_precision}")
        print("-" * 65)
        print("PREPROCESSING:")
        print(f"  CLAHE:              clip={self.clahe_clip_limit}, tile={self.clahe_tile_size}")
        print(f"  Bilateral Filter:   {self.use_bilateral_filter}")
        print("-" * 65)
        print("MODEL ARCHITECTURE:")
        print(f"  Base Model:         EfficientNet-B4")
        print(f"  Input Size:         {self.image_size[0]}x{self.image_size[1]}")
        print(f"  Dense Units:        {self.dense_units}")
        print(f"  Dropout:            {self.dropout_rate}")
        print(f"  Spatial Dropout:    {self.spatial_dropout_rate}")
        print(f"  L2 Regularization:  {self.l2_regularization}")
        print("-" * 65)
        print("TRAINING SCHEDULE:")
        print(f"  Batch Size:         {self.batch_size}")
        print(f"  Ensemble Models:    {self.ensemble_size}")
        print(f"  Stage 1:            {self.stage1_epochs} epochs @ LR={self.stage1_lr}")
        print(f"  Stage 2 (FOCUS):    {self.stage2_epochs} epochs @ LR={self.stage2_lr}")
        print(f"  Stage 3:            {self.stage3_epochs} epochs @ LR={self.stage3_lr}")
        print(f"  Total Epochs:       {total_training_epochs}")
        print("-" * 65)
        print("UNFREEZING STRATEGY:")
        print(f"  Stage 2 frozen:     {self.stage2_frozen_layers} layers")
        print(f"  Stage 3 frozen:     {self.stage3_frozen_layers} layers")
        print("-" * 65)
        print("AUGMENTATION:")
        print(f"  MixUp:              {self.use_mixup} (alpha={self.mixup_alpha})")
        print(f"  CutMix:             {self.use_cutmix} (alpha={self.cutmix_alpha})")
        print(f"  Random Erasing:     {self.use_random_erasing}")
        print(f"  Rotation:           ±{self.rotation_range}°")
        print(f"  TTA Augmentations:  {self.tta_augmentations}")
        print("-" * 65)
        print("CLASS BALANCE:")
        print(f"  Focal Alpha:        {self.focal_alpha}")
        print(f"  Weight Multiplier:  {self.malignant_weight_multiplier}")
        print("=" * 65)


config = ExperimentConfig(gpu_memory_gb=GPU_MEMORY_GB)
config.display()

if GPU_AVAILABLE and config.use_mixed_precision:
    try:
        policy = tf.keras.mixed_precision.Policy('mixed_float16')
        tf.keras.mixed_precision.set_global_policy(policy)
        print("✅ Mixed precision (FP16) enabled - 2x throughput!")
    except Exception as e:
        config.use_mixed_precision = False
        print(f"⚠️ Mixed precision not available: {e}")

## 3. Enhanced ROI Data Preprocessing

**V12 Pro Preprocessing Pipeline:**
1. DICOM loading with photometric correction
2. Optional bilateral filtering for noise reduction
3. CLAHE enhancement with optimized parameters
4. High-quality resizing with INTER_LANCZOS4
5. Proper [0, 255] range for EfficientNet compatibility

In [ ]:
def apply_clahe_enhanced(image, clip_limit=2.5, tile_size=(8, 8)):
    """
    Enhanced CLAHE preprocessing for mammography ROI images.
    
    Returns image in [0, 255] range for EfficientNet preprocessing.
    
    Args:
        image: Input grayscale image (any dtype)
        clip_limit: CLAHE threshold (higher = more contrast)
        tile_size: Grid size for local histogram equalization
    
    Returns:
        Enhanced image in [0, 255] range as float32
    """
    # Normalize to uint8 for CLAHE
    if image.dtype != np.uint8:
        img_min, img_max = image.min(), image.max()
        if img_max - img_min > 0:
            image = ((image - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        else:
            image = np.zeros_like(image, dtype=np.uint8)
    
    # Apply CLAHE
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    enhanced = clahe.apply(image)
    
    return enhanced.astype(np.float32)


def apply_preprocessing_pipeline(image, config):
    """
    Full preprocessing pipeline for mammography images.
    
    Pipeline:
    1. Bilateral filter (optional) - edge-preserving noise reduction
    2. CLAHE - adaptive contrast enhancement
    
    Args:
        image: Grayscale image as uint8
        config: ExperimentConfig instance
    
    Returns:
        Preprocessed image in [0, 255] range as float32
    """
    # Convert to uint8 if needed
    if image.dtype != np.uint8:
        img_min, img_max = image.min(), image.max()
        if img_max - img_min > 0:
            image = ((image - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        else:
            image = np.zeros_like(image, dtype=np.uint8)
    
    # Step 1: Optional bilateral filtering for noise reduction
    if config.use_bilateral_filter:
        image = cv2.bilateralFilter(
            image,
            d=config.bilateral_d,
            sigmaColor=config.bilateral_sigma_color,
            sigmaSpace=config.bilateral_sigma_space
        )
    
    # Step 2: CLAHE enhancement
    if config.use_clahe:
        clahe = cv2.createCLAHE(
            clipLimit=config.clahe_clip_limit,
            tileGridSize=config.clahe_tile_size
        )
        image = clahe.apply(image)
    
    return image.astype(np.float32)


def find_dicom_file(case_dir_name, base_path):
    """
    Locate DICOM file within CBIS-DDSM directory structure.
    
    Structure: case_dir / date_folder / series_folder / file.dcm
    """
    case_path = base_path / case_dir_name
    
    if not case_path.exists():
        return None
    
    try:
        for date_folder in case_path.iterdir():
            if date_folder.is_dir():
                for series_folder in date_folder.iterdir():
                    if series_folder.is_dir():
                        for dcm_file in series_folder.iterdir():
                            if dcm_file.suffix == '.dcm':
                                return dcm_file
    except Exception:
        pass
    
    return None


def load_dicom_image_enhanced(dicom_path, config):
    """
    Load and preprocess DICOM image with enhanced pipeline.
    
    Args:
        dicom_path: Path to DICOM file
        config: ExperimentConfig with preprocessing settings
    
    Returns:
        Preprocessed image (H, W, 3) in [0, 255] range or None
    """
    try:
        dcm = pydicom.dcmread(str(dicom_path))
        image = dcm.pixel_array.astype(np.float32)
        
        # Handle photometric interpretation
        if hasattr(dcm, 'PhotometricInterpretation'):
            if dcm.PhotometricInterpretation == 'MONOCHROME1':
                # Invert - brighter regions should be denser tissue
                image = image.max() - image
        
        # Normalize to uint8 for preprocessing
        img_min, img_max = image.min(), image.max()
        if img_max - img_min > 0:
            image = ((image - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        else:
            image = np.zeros_like(image, dtype=np.uint8)
        
        # Apply full preprocessing pipeline
        image = apply_preprocessing_pipeline(image, config)
        
        # High-quality resize to target size
        image = cv2.resize(
            image, 
            config.image_size, 
            interpolation=cv2.INTER_LANCZOS4
        )
        
        # Convert grayscale to 3-channel
        image = np.stack([image] * 3, axis=-1)
        
        return image
        
    except Exception as e:
        return None


def preprocess_roi_dataset(config, csv_files, split_name='train'):
    """
    Preprocess ROI images from CBIS-DDSM with enhanced pipeline.
    
    Args:
        config: ExperimentConfig instance
        csv_files: List of CSV file paths
        split_name: Data split identifier
    
    Returns:
        tuple: (images array in [0,255], labels array)
    """
    images = []
    labels = []
    processed_cases = set()
    stats = {'loaded': 0, 'skipped_no_path': 0, 'skipped_no_file': 0, 'skipped_error': 0}
    
    for csv_file in csv_files:
        if not csv_file.exists():
            print(f"  CSV not found: {csv_file}")
            continue
        
        df = pd.read_csv(csv_file)
        
        # Find path column
        path_col = None
        for col_name in ['cropped image file path', 'ROI mask file path', 'image file path']:
            if col_name in df.columns:
                path_col = col_name
                break
        
        if path_col is None:
            print(f"  No path column in {csv_file.name}")
            continue
        
        print(f"  Processing {csv_file.name} ({len(df)} rows)...")
        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"  {csv_file.name}"):
            # Get pathology label
            pathology = str(row.get('pathology', '')).upper()
            if 'MALIGNANT' in pathology:
                label = 1
            elif 'BENIGN' in pathology:
                label = 0
            else:
                continue
            
            # Get relative path
            rel_path = row.get(path_col, None)
            if pd.isna(rel_path) or rel_path is None:
                stats['skipped_no_path'] += 1
                continue
            
            # Extract case directory name
            case_dir_name = Path(rel_path).parts[0]
            
            # Skip duplicates
            case_key = (case_dir_name, label)
            if case_key in processed_cases:
                continue
            processed_cases.add(case_key)
            
            # Find DICOM file
            dicom_file = find_dicom_file(case_dir_name, config.dicom_path)
            if dicom_file is None:
                stats['skipped_no_file'] += 1
                continue
            
            # Load and preprocess
            image = load_dicom_image_enhanced(dicom_file, config)
            
            if image is not None:
                images.append(image)
                labels.append(label)
                stats['loaded'] += 1
            else:
                stats['skipped_error'] += 1
    
    print(f"\n  Preprocessing complete:")
    print(f"    Loaded: {stats['loaded']} images")
    print(f"    Skipped (no path): {stats['skipped_no_path']}")
    print(f"    Skipped (file not found): {stats['skipped_no_file']}")
    print(f"    Skipped (load error): {stats['skipped_error']}")
    
    if len(images) == 0:
        return np.array([]), np.array([])
    
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.float32)


def create_roi_cache(config):
    """Create preprocessed ROI cache with enhanced pipeline."""
    print("\n" + "=" * 60)
    print("CREATING ENHANCED ROI CACHE (V12 Pro)")
    print("=" * 60)
    print(f"Preprocessing: CLAHE(clip={config.clahe_clip_limit})")
    print(f"Bilateral Filter: {config.use_bilateral_filter}")
    print(f"Target size: {config.image_size}")
    
    # Process training data
    print("\nProcessing TRAINING data...")
    train_csvs = [config.calc_train_csv, config.mass_train_csv]
    train_images, train_labels = preprocess_roi_dataset(config, train_csvs, 'train')
    
    # Process test data
    print("\nProcessing TEST data...")
    test_csvs = [config.calc_test_csv, config.mass_test_csv]
    test_images, test_labels = preprocess_roi_dataset(config, test_csvs, 'test')
    
    if len(train_images) == 0:
        print("\n❌ ERROR: No training images found!")
        return
    
    # Split training into train/val
    val_ratio = config.val_split / (config.train_split + config.val_split)
    
    train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
        train_images, train_labels,
        test_size=val_ratio,
        random_state=RANDOM_SEED,
        stratify=train_labels
    )
    
    # Save cache
    np.savez_compressed(config.cache_path / 'train_cache.npz', images=train_imgs, labels=train_lbls)
    np.savez_compressed(config.cache_path / 'val_cache.npz', images=val_imgs, labels=val_lbls)
    np.savez_compressed(config.cache_path / 'test_cache.npz', images=test_images, labels=test_labels)
    
    # Calculate class distribution
    train_mal = (train_lbls == 1).sum()
    val_mal = (val_lbls == 1).sum()
    test_mal = (test_labels == 1).sum()
    
    print("\n" + "=" * 60)
    print("CACHE CREATED SUCCESSFULLY")
    print("=" * 60)
    print(f"Training:   {len(train_imgs)} samples (Mal: {train_mal}, Ben: {len(train_lbls)-train_mal})")
    print(f"Validation: {len(val_imgs)} samples (Mal: {val_mal}, Ben: {len(val_lbls)-val_mal})")
    print(f"Test:       {len(test_images)} samples (Mal: {test_mal}, Ben: {len(test_labels)-test_mal})")
    print(f"Cache path: {config.cache_path}")
    print("=" * 60)


print("✅ Enhanced preprocessing functions defined")

## 4. Data Loading

Load preprocessed ROI data from cache. Images are in [0, 255] range.

In [ ]:
def remount_drive():
    """Remount Google Drive if connection is lost."""
    if RUNTIME_ENV != "colab":
        return False

    try:
        from google.colab import drive
        print("Google Drive connection lost. Remounting...")
        try:
            drive.flush_and_unmount()
        except Exception:
            pass  # Ignore if already unmounted
        import time
        time.sleep(2)
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive remounted successfully")
        return True
    except Exception as e:
        print(f"Failed to remount: {e}")
        return False


def load_cached_data(config, max_retries=2):
    """
    Load preprocessed ROI data from cache files.

    Images are in [0, 255] range for EfficientNet preprocessing.
    Added retry logic for Google Drive connection issues.
    """
    cache_files = {
        'train': config.cache_path / 'train_cache.npz',
        'val': config.cache_path / 'val_cache.npz',
        'test': config.cache_path / 'test_cache.npz'
    }

    if not all(f.exists() for f in cache_files.values()):
        print("Cache not found. Creating from DICOM files...")
        create_roi_cache(config)

    if not all(f.exists() for f in cache_files.values()):
        raise FileNotFoundError(
            f"Cache files not found in {config.cache_path}.\n"
            f"Ensure DICOM data exists at: {config.dicom_path}"
        )

    # Initialize to None for proper cleanup on retry
    train_images = train_labels = None
    val_images = val_labels = None
    test_images = test_labels = None

    for attempt in range(max_retries + 1):
        try:
            # Use context managers (with statements) for proper file handle cleanup
            # This ensures files are closed even if an error occurs
            with np.load(cache_files['train']) as train_data:
                train_images = train_data['images'].copy()
                train_labels = train_data['labels'].copy()

            with np.load(cache_files['val']) as val_data:
                val_images = val_data['images'].copy()
                val_labels = val_data['labels'].copy()

            with np.load(cache_files['test']) as test_data:
                test_images = test_data['images'].copy()
                test_labels = test_data['labels'].copy()

            # Success - break out of retry loop
            break

        except OSError as e:
            error_msg = str(e)
            if "Transport endpoint is not connected" in error_msg or "Input/output error" in error_msg:
                if attempt < max_retries:
                    print(f"Drive connection error (attempt {attempt + 1}/{max_retries + 1})")
                    # Force garbage collection to clean up any dangling file handles
                    gc.collect()
                    if remount_drive():
                        continue
                raise OSError(
                    f"Google Drive connection lost after {max_retries + 1} attempts.\n"
                    f"Please remount manually:\n"
                    f"  from google.colab import drive\n"
                    f"  drive.mount('/content/drive', force_remount=True)"
                )
            else:
                raise

    print(f"Loaded: Train={len(train_images)}, Val={len(val_images)}, Test={len(test_images)}")

    return (train_images, train_labels, val_images, val_labels,
            test_images, test_labels)


train_images, train_labels, val_images, val_labels, test_images, test_labels = load_cached_data(config)

## 5. Dataset Pipeline with Advanced Augmentation

**V12 Pro Augmentation Pipeline:**
- Standard: flip, brightness, contrast
- MixUp: linear interpolation between samples
- CutMix: spatial mixing of samples
- Random Erasing: occlusion robustness
- Rotation: ±15° for orientation invariance
- EfficientNet normalization: [0,255] → [-1,1]

In [ ]:
# ============================================================
# ADVANCED AUGMENTATION FUNCTIONS (V12 Pro)
# ============================================================
# 
# V12 Pro Augmentation Strategy:
# - Basic augmentations (flip, brightness, contrast) are applied during training
# - TTA handles rotations (90°, 180°, 270°) to avoid interpolation artifacts
# - MixUp/CutMix are defined but used sparingly due to medical imaging specifics
# ============================================================

@tf.function
def augment_basic(images):
    """
    Basic GPU-accelerated augmentation on [0, 255] range images.
    
    These augmentations are safe for medical imaging as they don't introduce
    interpolation artifacts that could affect diagnostic features.
    """
    images = tf.image.random_flip_left_right(images)
    images = tf.image.random_flip_up_down(images)
    images = tf.image.random_brightness(images, max_delta=25.5)
    images = tf.image.random_contrast(images, lower=0.85, upper=1.15)
    images = tf.clip_by_value(images, 0.0, 255.0)
    return images


@tf.function
def random_erasing(images, probability=0.25, scale_min=0.02, scale_max=0.15):
    """
    Random Erasing augmentation - masks random rectangular regions with mean pixel value.
    
    Helps model learn to classify even with partial occlusion/missing data.
    Uses TensorFlow ops for compatibility with tf.data pipeline.
    
    Args:
        images: Batch of images [B, H, W, C] in [0, 255]
        probability: Chance of applying erasing (per image)
        scale_min: Minimum area ratio to erase
        scale_max: Maximum area ratio to erase
    
    Returns:
        Augmented images with random rectangular regions replaced by mean value
    """
    # Get dimensions
    shape = tf.shape(images)
    batch_size = shape[0]
    height = shape[1]
    width = shape[2]
    
    height_f = tf.cast(height, tf.float32)
    width_f = tf.cast(width, tf.float32)
    
    # Create coordinate grids [height, width]
    y_coords = tf.range(height, dtype=tf.float32)  # [H]
    x_coords = tf.range(width, dtype=tf.float32)   # [W]
    
    # Random parameters for each image in batch
    apply_mask = tf.random.uniform([batch_size]) < probability
    area_ratio = tf.random.uniform([batch_size], scale_min, scale_max)
    
    # Calculate erase dimensions
    erase_h = tf.sqrt(area_ratio) * height_f  # [B]
    erase_w = tf.sqrt(area_ratio) * width_f   # [B]
    
    # Random top-left corner
    max_top = height_f - erase_h
    max_left = width_f - erase_w
    top = tf.random.uniform([batch_size]) * tf.maximum(max_top, 1.0)
    left = tf.random.uniform([batch_size]) * tf.maximum(max_left, 1.0)
    
    # Build masks for each image
    # Shape: [B, H, W]
    y_expanded = tf.reshape(y_coords, [1, height, 1])    # [1, H, 1]
    x_expanded = tf.reshape(x_coords, [1, 1, width])     # [1, 1, W]
    
    top_expanded = tf.reshape(top, [batch_size, 1, 1])        # [B, 1, 1]
    left_expanded = tf.reshape(left, [batch_size, 1, 1])      # [B, 1, 1]
    erase_h_expanded = tf.reshape(erase_h, [batch_size, 1, 1]) # [B, 1, 1]
    erase_w_expanded = tf.reshape(erase_w, [batch_size, 1, 1]) # [B, 1, 1]
    
    # Create binary mask: 1 where we should erase
    in_box_y = tf.logical_and(y_expanded >= top_expanded, 
                               y_expanded < top_expanded + erase_h_expanded)
    in_box_x = tf.logical_and(x_expanded >= left_expanded, 
                               x_expanded < left_expanded + erase_w_expanded)
    in_box = tf.logical_and(in_box_y, in_box_x)  # [B, H, W]
    
    # Apply only to selected images
    apply_mask_expanded = tf.reshape(apply_mask, [batch_size, 1, 1])
    final_mask = tf.logical_and(in_box, apply_mask_expanded)  # [B, H, W]
    
    # Expand mask to [B, H, W, C]
    final_mask_4d = tf.cast(tf.expand_dims(final_mask, -1), tf.float32)
    
    # Fill value: mean pixel value of the image (128 as reasonable default)
    fill_value = 128.0
    
    # Apply mask: keep original where mask=0, fill with mean where mask=1
    result = images * (1.0 - final_mask_4d) + fill_value * final_mask_4d
    
    return result


def mixup_tf(images, labels, alpha=0.2):
    """
    MixUp augmentation using pure TensorFlow operations.
    
    Creates virtual training examples by blending image pairs.
    Compatible with tf.data pipeline.
    
    Args:
        images: Batch of images [B, H, W, C]
        labels: Batch of labels [B]
        alpha: Beta distribution parameter (higher = more mixing)
    
    Returns:
        Mixed images and labels
    """
    batch_size = tf.shape(images)[0]
    
    # Sample lambda from Beta distribution using TensorFlow
    # Approximation: use uniform and transform (simpler than true Beta)
    # For alpha=0.2, Beta(0.2, 0.2) is U-shaped, often close to 0 or 1
    u = tf.random.uniform([])
    # Kumaraswamy approximation to Beta for speed
    lam = tf.pow(u, 1.0 / alpha) if alpha > 0 else 0.5
    lam = tf.maximum(lam, 1.0 - lam)  # Ensure lam >= 0.5 for stability
    
    # Random permutation for pairing
    indices = tf.random.shuffle(tf.range(batch_size))
    
    # Mix images and labels
    shuffled_images = tf.gather(images, indices)
    shuffled_labels = tf.gather(labels, indices)
    
    mixed_images = lam * images + (1.0 - lam) * shuffled_images
    mixed_labels = lam * labels + (1.0 - lam) * shuffled_labels
    
    return mixed_images, mixed_labels


@tf.function
def apply_efficientnet_preprocessing(images):
    """
    Apply EfficientNet-specific preprocessing.
    
    Converts [0, 255] → [-1, 1] using (x - 127.5) / 127.5
    This is the expected input range for EfficientNet models.
    """
    return tf.keras.applications.efficientnet.preprocess_input(images)


def create_generator(images, labels, shuffle=False, seed=None):
    """Generator function for tf.data.Dataset."""
    indices = np.arange(len(images))
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
    
    for idx in indices:
        yield images[idx], labels[idx]


def create_dataset(images, labels, config, training=True,
                   shuffle_seed=None, use_reduced_batch=False):
    """
    Create tf.data.Dataset with V12 Pro augmentation pipeline.
    
    Training pipeline:
    1. Load from generator
    2. Shuffle (if training)
    3. Batch
    4. Basic augmentation (flip, brightness, contrast)
    5. EfficientNet preprocessing
    6. Prefetch
    
    Note: MixUp is available but not applied by default as medical imaging
    often benefits more from geometric augmentations.
    """
    output_signature = (
        tf.TensorSpec(shape=(config.image_size[0], config.image_size[1], 3),
                      dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: create_generator(images, labels, shuffle=training, seed=shuffle_seed),
        output_signature=output_signature
    )
    
    if training:
        dataset = dataset.shuffle(config.shuffle_buffer)
    
    batch_size = config.batch_size_finetune if use_reduced_batch else config.batch_size
    dataset = dataset.batch(batch_size, drop_remainder=training)
    
    if training:
        # Basic augmentation (flip, brightness, contrast)
        dataset = dataset.map(
            lambda x, y: (augment_basic(x), y),
            num_parallel_calls=config.num_parallel_calls
        )
        
        # Optional: Random erasing (controlled by config)
        if hasattr(config, 'use_random_erasing') and config.use_random_erasing:
            dataset = dataset.map(
                lambda x, y: (random_erasing(x, probability=0.15), y),
                num_parallel_calls=config.num_parallel_calls
            )
    
    # Apply EfficientNet preprocessing (converts [0,255] to [-1,1])
    dataset = dataset.map(
        lambda x, y: (apply_efficientnet_preprocessing(x), y),
        num_parallel_calls=config.num_parallel_calls
    )
    
    dataset = dataset.prefetch(config.prefetch_buffer)
    
    return dataset


def create_dataset_with_mixup(images, labels, config, shuffle_seed=None, mixup_prob=0.5):
    """
    Create training dataset with MixUp augmentation applied probabilistically.
    
    Note: MixUp modifies labels, so use with caution for medical imaging
    where class boundaries should be clear.
    
    Args:
        images: Training images
        labels: Training labels
        config: ExperimentConfig
        shuffle_seed: Random seed for shuffling
        mixup_prob: Probability of applying MixUp per batch
    
    Returns:
        tf.data.Dataset with MixUp applied probabilistically
    """
    # Create base dataset without final preprocessing
    output_signature = (
        tf.TensorSpec(shape=(config.image_size[0], config.image_size[1], 3),
                      dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.float32)
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: create_generator(images, labels, shuffle=True, seed=shuffle_seed),
        output_signature=output_signature
    )
    
    dataset = dataset.shuffle(config.shuffle_buffer)
    dataset = dataset.batch(config.batch_size, drop_remainder=True)
    
    # Basic augmentation
    dataset = dataset.map(
        lambda x, y: (augment_basic(x), y),
        num_parallel_calls=config.num_parallel_calls
    )
    
    # Probabilistic MixUp
    def maybe_mixup(images, labels):
        apply = tf.random.uniform([]) < mixup_prob
        mixed_images, mixed_labels = tf.cond(
            apply,
            lambda: mixup_tf(images, labels, alpha=config.mixup_alpha),
            lambda: (images, labels)
        )
        return mixed_images, mixed_labels
    
    dataset = dataset.map(maybe_mixup, num_parallel_calls=config.num_parallel_calls)
    
    # EfficientNet preprocessing
    dataset = dataset.map(
        lambda x, y: (apply_efficientnet_preprocessing(x), y),
        num_parallel_calls=config.num_parallel_calls
    )
    
    dataset = dataset.prefetch(config.prefetch_buffer)
    
    return dataset


# Create datasets
print("Creating datasets...")
train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

print(f"✅ Datasets created")
print(f"   Training batches: ~{len(train_images) // config.batch_size}")
print(f"   Validation batches: ~{len(val_images) // config.batch_size}")
print(f"   Test batches: ~{len(test_images) // config.batch_size}")

## 6. Class Weight Computation

Compute balanced class weights for handling class imbalance in CBIS-DDSM dataset.

In [ ]:
def compute_class_weights(labels, malignant_multiplier=2.5):
    """Compute class weights for imbalanced dataset."""
    balanced = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=labels
    )

    weights = {
        0: balanced[0],
        1: balanced[1] * malignant_multiplier
    }

    return weights


config.class_weights = compute_class_weights(
    train_labels,
    config.malignant_weight_multiplier
)

print(f"Class Weights: Benign={config.class_weights[0]:.3f}, Malignant={config.class_weights[1]:.3f}")

## 7. Model Architecture (V12 Pro)

**Architecture Enhancements:**
- EfficientNet-B4 backbone with ImageNet weights
- Spatial Dropout after GlobalAveragePooling (new)
- Dual Dense layers for better feature extraction
- Stronger L2 regularization
- Mixed precision compatible output

In [ ]:
def create_focal_loss(gamma=2.0, alpha=0.65):
    """
    Focal Loss for handling class imbalance.
    
    V12 Pro: Reduced alpha (0.65) to decrease malignant bias.
    
    Args:
        gamma: Focusing parameter (higher = more focus on hard examples)
        alpha: Class balancing factor (weight for positive class)
    """
    @tf.function
    def focal_loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = tf.pow(1 - p_t, gamma)
        
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        
        ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        
        return tf.reduce_mean(alpha_t * focal_weight * ce)
    
    return focal_loss


class SpatialDropout1D(layers.Layer):
    """
    Spatial Dropout - drops entire feature maps.
    
    More effective than standard dropout for convolutional features
    because it maintains spatial coherence.
    """
    def __init__(self, rate, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate
        self.dropout = layers.Dropout(rate)
    
    def call(self, inputs, training=None):
        if training:
            # Reshape to add spatial dimension, apply dropout, reshape back
            shape = tf.shape(inputs)
            # For 2D inputs after GAP: (batch, features)
            x = tf.expand_dims(inputs, axis=1)  # (batch, 1, features)
            x = self.dropout(x, training=training)
            x = tf.squeeze(x, axis=1)
            return x
        return inputs


def build_model_v12pro(config, seed=42):
    """
    Build EfficientNet-B4 V12 Pro model with enhanced architecture.
    
    Architecture:
    - EfficientNet-B4 backbone (frozen initially)
    - GlobalAveragePooling2D
    - SpatialDropout (new)
    - Dense(1024) + BN + ReLU + Dropout
    - Dense(256) + BN + ReLU + Dropout (new intermediate layer)
    - Dense(1) sigmoid output
    
    Args:
        config: ExperimentConfig
        seed: Random seed
    
    Returns:
        Compiled Keras model
    """
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
    inputs = layers.Input(shape=(config.image_size[0], config.image_size[1], 3))
    
    # EfficientNet-B4 backbone
    base_model = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        pooling=None
    )
    
    # Freeze backbone initially
    base_model.trainable = False
    
    x = base_model.output
    
    # Global Average Pooling
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    
    # Spatial Dropout (new in V12 Pro)
    if config.use_spatial_dropout:
        x = layers.Dropout(config.spatial_dropout_rate, name='spatial_dropout')(x)
    
    # First Dense block
    x = layers.Dense(
        config.dense_units,
        kernel_regularizer=regularizers.l2(config.l2_regularization),
        kernel_initializer='he_normal',
        name='dense_1'
    )(x)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Activation('relu', name='relu_1')(x)
    x = layers.Dropout(config.dropout_rate, name='dropout_1')(x)
    
    # Second Dense block (new in V12 Pro - intermediate layer)
    intermediate_units = max(config.dense_units // 4, 128)
    x = layers.Dense(
        intermediate_units,
        kernel_regularizer=regularizers.l2(config.l2_regularization),
        kernel_initializer='he_normal',
        name='dense_2'
    )(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Activation('relu', name='relu_2')(x)
    x = layers.Dropout(config.dropout_rate * 0.5, name='dropout_2')(x)  # Less dropout here
    
    # Output layer (float32 for mixed precision compatibility)
    outputs = layers.Dense(
        1,
        activation='sigmoid',
        dtype='float32',
        name='output'
    )(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='EfficientNetB4_V12Pro')
    
    # Compile with focal loss
    if config.use_focal_loss:
        loss = create_focal_loss(config.focal_gamma, config.focal_alpha)
    else:
        loss = BinaryCrossentropy(label_smoothing=config.label_smoothing)
    
    model.compile(
        optimizer=Adam(
            learning_rate=config.stage1_lr,
            clipnorm=config.gradient_clip_norm
        ),
        loss=loss,
        metrics=[
            BinaryAccuracy(name='accuracy'),
            AUC(name='auc'),
            Precision(name='precision'),
            Recall(name='recall')
        ]
    )
    
    return model


def unfreeze_layers(model, num_frozen_layers):
    """Unfreeze backbone layers from specified index."""
    for i, layer in enumerate(model.layers):
        if i < num_frozen_layers:
            layer.trainable = False
        else:
            layer.trainable = True


def get_trainable_summary(model):
    """Get trainable parameter counts."""
    trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
    non_trainable = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
    return trainable, non_trainable


def count_model_layers(model):
    """Count total layers in model."""
    return len(model.layers)


# Build test model
print("Building V12 Pro model architecture...")
test_model = build_model_v12pro(config, seed=42)

trainable, non_trainable = get_trainable_summary(test_model)
total_layers = count_model_layers(test_model)

print("\n" + "=" * 60)
print("V12 PRO MODEL ARCHITECTURE")
print("=" * 60)
print(f"Total parameters:     {trainable + non_trainable:,}")
print(f"Trainable:            {trainable:,}")
print(f"Non-trainable:        {non_trainable:,}")
print(f"Total layers:         {total_layers}")
print(f"Dense layer 1:        {config.dense_units} units")
print(f"Dense layer 2:        {max(config.dense_units // 4, 128)} units")
print(f"Dropout rate:         {config.dropout_rate}")
print(f"Spatial dropout:      {config.spatial_dropout_rate}")
print("=" * 60)

# Cleanup
del test_model
gc.collect()

## 8. Training Callbacks (V12 Pro)

**Enhanced Callbacks:**
- Warmup + Cosine Annealing LR schedule
- Min-epoch early stopping (Stage 2 focus)
- Best checkpoint tracking across ALL stages (new)
- Per-stage checkpointing with AUC monitoring

In [ ]:
class WarmupCosineScheduler(Callback):
    """
    Learning rate scheduler with warmup and cosine annealing.
    
    V12 Pro: Combines warmup (first few epochs) with cosine decay
    for smoother convergence.
    """
    
    def __init__(self, warmup_epochs, total_epochs, start_lr, target_lr, min_lr=1e-7):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.start_lr = start_lr
        self.target_lr = target_lr
        self.min_lr = min_lr
    
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            # Linear warmup
            lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
        else:
            # Cosine annealing
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + (self.target_lr - self.min_lr) * 0.5 * (1 + np.cos(np.pi * progress))
        
        self.model.optimizer.learning_rate.assign(lr)


class MinEpochEarlyStopping(Callback):
    """
    Early stopping with minimum epoch requirement.
    
    Prevents stopping too early before model has had chance to learn.
    """
    
    def __init__(self, monitor='val_auc', patience=20, min_epochs=30, mode='max', 
                 restore_best_weights=True):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.min_epochs = min_epochs
        self.mode = mode
        self.restore_best = restore_best_weights
        self.wait = 0
        self.best = -np.inf if mode == 'max' else np.inf
        self.best_weights = None
        self.best_epoch = 0
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if self._is_improvement(current):
            self.best = current
            self.best_epoch = epoch
            self.wait = 0
            if self.restore_best:
                self.best_weights = self.model.get_weights()
        else:
            if epoch >= self.min_epochs:
                self.wait += 1
                if self.wait >= self.patience:
                    self.model.stop_training = True
                    print(f"\n⏹️ Early stopping at epoch {epoch + 1} (best: {self.best:.4f} @ epoch {self.best_epoch + 1})")
                    if self.restore_best and self.best_weights is not None:
                        self.model.set_weights(self.best_weights)
    
    def _is_improvement(self, current):
        if self.mode == 'max':
            return current > self.best
        return current < self.best


class BestCheckpointTracker:
    """
    Track best checkpoint across ALL stages for a model.
    
    V12 Pro: Since V12/V13 showed Stage 2 often produces best results,
    we track the best checkpoint across all stages and restore it at the end.
    """
    
    def __init__(self, checkpoint_dir, model_idx):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.model_idx = model_idx
        self.best_val_auc = -np.inf
        self.best_stage = None
        self.best_epoch = None
        self.best_checkpoint_path = None
    
    def update(self, val_auc, stage, epoch, model):
        """Update best checkpoint if current is better."""
        if val_auc > self.best_val_auc:
            self.best_val_auc = val_auc
            self.best_stage = stage
            self.best_epoch = epoch
            
            # Save as global best
            self.best_checkpoint_path = self.checkpoint_dir / f'model_{self.model_idx}_global_best.keras'
            model.save(str(self.best_checkpoint_path))
            
            print(f"   🌟 New best: AUC={val_auc:.4f} (Stage {stage}, Epoch {epoch + 1})")
            return True
        return False
    
    def get_best_path(self):
        """Get path to best checkpoint."""
        return self.best_checkpoint_path
    
    def summary(self):
        """Print summary of best checkpoint."""
        return f"Stage {self.best_stage}, Epoch {self.best_epoch + 1}, AUC={self.best_val_auc:.4f}"


def get_callbacks_v12pro(config, stage, model_idx, total_epochs):
    """
    Create callbacks for V12 Pro training stage.
    
    Args:
        config: ExperimentConfig
        stage: Training stage (1, 2, or 3)
        model_idx: Ensemble model index
        total_epochs: Total epochs for this stage
    
    Returns:
        List of callbacks
    """
    callbacks = []
    
    # Get stage-specific settings
    if stage == 1:
        target_lr = config.stage1_lr
        min_epochs = config.min_epochs_stage1
    elif stage == 2:
        target_lr = config.stage2_lr
        min_epochs = config.min_epochs_stage2
    else:
        target_lr = config.stage3_lr
        min_epochs = config.min_epochs_stage3
    
    # Warmup + Cosine Annealing (Stage 1 only has warmup)
    if config.use_cosine_annealing:
        callbacks.append(WarmupCosineScheduler(
            warmup_epochs=config.warmup_epochs if stage == 1 else 2,
            total_epochs=total_epochs,
            start_lr=config.warmup_start_lr if stage == 1 else target_lr * 0.1,
            target_lr=target_lr,
            min_lr=config.cosine_min_lr
        ))
    elif stage == 1 and config.use_lr_warmup:
        # Fallback to simple warmup
        callbacks.append(WarmupCosineScheduler(
            warmup_epochs=config.warmup_epochs,
            total_epochs=total_epochs,
            start_lr=config.warmup_start_lr,
            target_lr=target_lr,
            min_lr=target_lr  # No decay
        ))
    
    # ReduceLROnPlateau as fallback
    callbacks.append(ReduceLROnPlateau(
        monitor='val_auc',
        factor=config.lr_reduce_factor,
        patience=config.lr_reduce_patience,
        min_lr=1e-7,
        mode='max',
        verbose=1
    ))
    
    # Stage checkpoint
    checkpoint_path = config.checkpoint_path / f'model_{model_idx}_stage{stage}_best.keras'
    callbacks.append(ModelCheckpoint(
        str(checkpoint_path),
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=0
    ))
    
    # Early stopping with min epochs
    callbacks.append(MinEpochEarlyStopping(
        monitor='val_auc',
        patience=config.early_stop_patience,
        min_epochs=min_epochs,
        mode='max',
        restore_best_weights=True
    ))
    
    return callbacks


print("✅ V12 Pro callback classes defined")

## 9. Training Functions (V12 Pro)

**Training Strategy:**
- Stage 1: Train classification head only (25 epochs)
- Stage 2: Fine-tune top layers - **FOCUS STAGE** (60 epochs)
- Stage 3: Conservative full fine-tune (20 epochs)
- Best checkpoint selected across ALL stages
- Memory-efficient ensemble training with disk offload

In [ ]:
def train_single_model_v12pro(config, train_ds, val_ds, train_steps, val_steps, model_idx=0):
    """
    Train single model with V12 Pro 3-stage progressive unfreezing.
    
    Key improvements:
    - Best checkpoint tracking across all stages
    - Extended Stage 2 (focus stage based on V12/V13 analysis)
    - Conservative Stage 3 with reduced epochs
    - Proper learning rate scheduling per stage
    
    Args:
        config: ExperimentConfig
        train_ds: Training dataset
        val_ds: Validation dataset
        train_steps: Steps per training epoch
        val_steps: Steps per validation epoch
        model_idx: Ensemble model index
    
    Returns:
        tuple: (trained model, combined history, best_tracker)
    """
    print(f"\n{'='*65}")
    print(f"🚀 TRAINING MODEL {model_idx + 1}/{config.ensemble_size} (V12 Pro)")
    print(f"{'='*65}")
    
    seed = config.ensemble_seeds[model_idx]
    model = build_model_v12pro(config, seed=seed)
    
    # Initialize best checkpoint tracker
    best_tracker = BestCheckpointTracker(config.checkpoint_path, model_idx)
    
    combined_history = {
        'loss': [], 'val_loss': [],
        'accuracy': [], 'val_accuracy': [],
        'auc': [], 'val_auc': [],
        'precision': [], 'val_precision': [],
        'recall': [], 'val_recall': [],
        'lr': [], 'stage': []
    }
    
    # ==================== STAGE 1 ====================
    print(f"\n{'─'*65}")
    print(f"📚 Stage 1: Training classification head ({config.stage1_epochs} epochs)")
    print(f"   LR: {config.stage1_lr}, Frozen: backbone")
    print(f"{'─'*65}")
    
    callbacks = get_callbacks_v12pro(config, stage=1, model_idx=model_idx, 
                                      total_epochs=config.stage1_epochs)
    
    history1 = model.fit(
        train_ds,
        epochs=config.stage1_epochs,
        steps_per_epoch=train_steps,
        validation_data=val_ds,
        validation_steps=val_steps,
        class_weight=config.class_weights,
        callbacks=callbacks,
        verbose=1
    )
    
    # Track history and best checkpoint
    for key in combined_history:
        if key in history1.history:
            combined_history[key].extend(history1.history[key])
        elif key == 'lr':
            combined_history['lr'].extend([float(model.optimizer.learning_rate)] * len(history1.history['loss']))
        elif key == 'stage':
            combined_history['stage'].extend([1] * len(history1.history['loss']))
    
    # Check for best across Stage 1
    for epoch, val_auc in enumerate(history1.history.get('val_auc', [])):
        best_tracker.update(val_auc, stage=1, epoch=epoch, model=model)
    
    gc.collect()
    
    # ==================== STAGE 2 (FOCUS) ====================
    print(f"\n{'─'*65}")
    print(f"🎯 Stage 2: Fine-tuning top layers - FOCUS STAGE ({config.stage2_epochs} epochs)")
    print(f"   LR: {config.stage2_lr}, Frozen: {config.stage2_frozen_layers} layers")
    print(f"{'─'*65}")
    
    # Unfreeze top layers
    unfreeze_layers(model, config.stage2_frozen_layers)
    
    trainable, _ = get_trainable_summary(model)
    print(f"   Trainable parameters: {trainable:,}")
    
    # Recompile with new learning rate
    if config.use_focal_loss:
        loss = create_focal_loss(config.focal_gamma, config.focal_alpha)
    else:
        loss = BinaryCrossentropy(label_smoothing=config.label_smoothing)
    
    model.compile(
        optimizer=Adam(learning_rate=config.stage2_lr, clipnorm=config.gradient_clip_norm),
        loss=loss,
        metrics=[
            BinaryAccuracy(name='accuracy'),
            AUC(name='auc'),
            Precision(name='precision'),
            Recall(name='recall')
        ]
    )
    
    callbacks = get_callbacks_v12pro(config, stage=2, model_idx=model_idx,
                                      total_epochs=config.stage2_epochs)
    
    history2 = model.fit(
        train_ds,
        epochs=config.stage2_epochs,
        steps_per_epoch=train_steps,
        validation_data=val_ds,
        validation_steps=val_steps,
        class_weight=config.class_weights,
        callbacks=callbacks,
        verbose=1
    )
    
    # Track history
    epoch_offset = len(combined_history['loss'])
    for key in combined_history:
        if key in history2.history:
            combined_history[key].extend(history2.history[key])
        elif key == 'lr':
            combined_history['lr'].extend([float(model.optimizer.learning_rate)] * len(history2.history['loss']))
        elif key == 'stage':
            combined_history['stage'].extend([2] * len(history2.history['loss']))
    
    # Check for best across Stage 2
    for epoch, val_auc in enumerate(history2.history.get('val_auc', [])):
        best_tracker.update(val_auc, stage=2, epoch=epoch_offset + epoch, model=model)
    
    gc.collect()
    
    # ==================== STAGE 3 (CONSERVATIVE) ====================
    print(f"\n{'─'*65}")
    print(f"🔬 Stage 3: Conservative full fine-tuning ({config.stage3_epochs} epochs)")
    print(f"   LR: {config.stage3_lr}, Frozen: {config.stage3_frozen_layers} layers")
    print(f"{'─'*65}")
    
    # More conservative unfreezing (keep some early layers frozen)
    unfreeze_layers(model, config.stage3_frozen_layers)
    
    trainable, _ = get_trainable_summary(model)
    print(f"   Trainable parameters: {trainable:,}")
    
    model.compile(
        optimizer=Adam(learning_rate=config.stage3_lr, clipnorm=config.gradient_clip_norm),
        loss=loss,
        metrics=[
            BinaryAccuracy(name='accuracy'),
            AUC(name='auc'),
            Precision(name='precision'),
            Recall(name='recall')
        ]
    )
    
    callbacks = get_callbacks_v12pro(config, stage=3, model_idx=model_idx,
                                      total_epochs=config.stage3_epochs)
    
    history3 = model.fit(
        train_ds,
        epochs=config.stage3_epochs,
        steps_per_epoch=train_steps,
        validation_data=val_ds,
        validation_steps=val_steps,
        class_weight=config.class_weights,
        callbacks=callbacks,
        verbose=1
    )
    
    # Track history
    epoch_offset = len(combined_history['loss'])
    for key in combined_history:
        if key in history3.history:
            combined_history[key].extend(history3.history[key])
        elif key == 'lr':
            combined_history['lr'].extend([float(model.optimizer.learning_rate)] * len(history3.history['loss']))
        elif key == 'stage':
            combined_history['stage'].extend([3] * len(history3.history['loss']))
    
    # Check for best across Stage 3
    for epoch, val_auc in enumerate(history3.history.get('val_auc', [])):
        best_tracker.update(val_auc, stage=3, epoch=epoch_offset + epoch, model=model)
    
    # ==================== LOAD BEST CHECKPOINT ====================
    print(f"\n{'─'*65}")
    print(f"📊 Training complete for Model {model_idx + 1}")
    print(f"   Best checkpoint: {best_tracker.summary()}")
    print(f"{'─'*65}")
    
    # Load the global best (across all stages)
    if config.select_best_across_stages and best_tracker.best_checkpoint_path:
        print(f"   Loading global best checkpoint...")
        model = tf.keras.models.load_model(
            str(best_tracker.best_checkpoint_path),
            compile=False
        )
        # Recompile for evaluation
        model.compile(
            optimizer=Adam(learning_rate=1e-5),
            loss=loss,
            metrics=[
                BinaryAccuracy(name='accuracy'),
                AUC(name='auc'),
                Precision(name='precision'),
                Recall(name='recall')
            ]
        )
    
    print(f"\n✅ Model {model_idx + 1} training complete!")
    print_memory_usage("After Training")
    
    return model, combined_history, best_tracker


def train_ensemble_v12pro(config, train_images, train_labels, val_images, val_labels):
    """
    Train V12 Pro ensemble with memory-efficient approach.
    
    Features:
    - Each model saved to disk after training
    - GPU memory cleared between models
    - Best checkpoint selection across all stages
    - Comprehensive history tracking
    """
    model_paths = []
    histories = []
    best_trackers = []
    
    train_steps = len(train_labels) // config.batch_size
    val_steps = max(1, len(val_labels) // config.batch_size)
    
    print("\n" + "=" * 65)
    print("🏋️ V12 PRO ENSEMBLE TRAINING")
    print("=" * 65)
    print(f"Ensemble size:        {config.ensemble_size}")
    print(f"Training samples:     {len(train_labels)}")
    print(f"Validation samples:   {len(val_labels)}")
    print(f"Batch size:           {config.batch_size}")
    print(f"Steps per epoch:      {train_steps}")
    print(f"Total epochs/model:   {config.stage1_epochs + config.stage2_epochs + config.stage3_epochs}")
    print("=" * 65)
    
    for model_idx in range(config.ensemble_size):
        seed = config.ensemble_seeds[model_idx]
        
        # Create fresh datasets with different shuffling
        train_ds = create_dataset(
            train_images, train_labels, config,
            training=True, shuffle_seed=seed
        )
        val_ds = create_dataset(
            val_images, val_labels, config,
            training=False
        )
        
        # Train the model
        model, history, tracker = train_single_model_v12pro(
            config, train_ds, val_ds,
            train_steps, val_steps, model_idx
        )
        
        # Save final model
        model_save_path = config.checkpoint_path / f'model_{model_idx}_final.keras'
        model.save(str(model_save_path))
        model_paths.append(model_save_path)
        histories.append(history)
        best_trackers.append(tracker)
        
        print(f"💾 Model {model_idx + 1} saved to {model_save_path.name}")
        
        # Clear memory before next model
        del model
        del train_ds
        del val_ds
        tf.keras.backend.clear_session()
        gc.collect()
        
        print_memory_usage(f"After Model {model_idx + 1}")
    
    # Summary
    print("\n" + "=" * 65)
    print("📈 ENSEMBLE TRAINING SUMMARY")
    print("=" * 65)
    for idx, tracker in enumerate(best_trackers):
        print(f"Model {idx + 1}: {tracker.summary()}")
    print("=" * 65)
    
    # Reload all models
    print("\n🔄 Reloading models for ensemble prediction...")
    models_list = []
    for idx, tracker in enumerate(best_trackers):
        # Load the global best checkpoint for each model
        best_path = tracker.get_best_path()
        if best_path and best_path.exists():
            print(f"   Loading Model {idx + 1} from: {best_path.name}")
            model = tf.keras.models.load_model(str(best_path), compile=False)
        else:
            # Fallback to final model
            print(f"   Loading Model {idx + 1} from final checkpoint")
            model = tf.keras.models.load_model(str(model_paths[idx]), compile=False)
        models_list.append(model)
    
    print(f"\n✅ Ensemble of {len(models_list)} model(s) ready for evaluation!")
    
    return models_list, histories, best_trackers

## 10. Execute Training

Run the full training pipeline.

In [ ]:
# ============================================================
# EXECUTE V12 PRO TRAINING
# ============================================================

total_epochs_per_model = config.stage1_epochs + config.stage2_epochs + config.stage3_epochs
total_epochs_all = total_epochs_per_model * config.ensemble_size

print("\n" + "=" * 65)
print("🏁 V12 PRO TRAINING CONFIGURATION")
print("=" * 65)
print(f"Ensemble Models:      {config.ensemble_size}")
print(f"Training Samples:     {len(train_labels)}")
print(f"Validation Samples:   {len(val_labels)}")
print(f"Test Samples:         {len(test_labels)}")
print("-" * 65)
print("Training Schedule:")
print(f"  Stage 1 (Head):     {config.stage1_epochs} epochs @ LR={config.stage1_lr}")
print(f"  Stage 2 (Focus):    {config.stage2_epochs} epochs @ LR={config.stage2_lr}")
print(f"  Stage 3 (Fine):     {config.stage3_epochs} epochs @ LR={config.stage3_lr}")
print(f"  Total per model:    {total_epochs_per_model} epochs")
print(f"  Grand total:        {total_epochs_all} epochs")
print("-" * 65)
print("Preprocessing:        CLAHE + Bilateral + EfficientNet norm")
print("Best checkpoint:      Selected across ALL stages")
print("=" * 65)
print()

# Train ensemble
ensemble_models, training_histories, best_trackers = train_ensemble_v12pro(
    config, train_images, train_labels, val_images, val_labels
)

print("\n✅ Training completed!")

## 11. Test-Time Augmentation (V12 Pro)

**Enhanced TTA Pipeline:**
- 8 augmentations: original + 7 geometric transforms
- Memory-efficient per-image processing
- Batch processing optimized for A100
- NaN handling for robustness

In [ ]:
def apply_tta_single(image):
    """
    Generate TTA augmented versions of a SINGLE image.

    Memory-optimized: processes one image at a time instead of entire dataset.
    Input image should be in [0, 255] range with shape (H, W, 3).
    Returns generator to avoid storing all augmentations in memory.

    Supports up to 8 augmentations:
    1. Original
    2. Horizontal flip
    3. Vertical flip
    4. Both flips
    5. 90° rotation
    6. 180° rotation
    7. 270° rotation
    8. 90° rotation + horizontal flip

    Args:
        image: Single image array (H, W, 3) in [0, 255] range

    Yields:
        Augmented image versions one at a time
    """
    yield image  # 1. Original
    yield np.flip(image, axis=1)  # 2. Horizontal flip
    yield np.flip(image, axis=0)  # 3. Vertical flip
    yield np.flip(np.flip(image, axis=1), axis=0)  # 4. Both flips
    yield np.rot90(image, k=1)  # 5. 90° rotation
    yield np.rot90(image, k=2)  # 6. 180° rotation
    yield np.rot90(image, k=3)  # 7. 270° rotation
    yield np.flip(np.rot90(image, k=1), axis=1)  # 8. 90° + horizontal flip


def predict_with_tta_memory_efficient(models, images, config):
    """
    Memory-efficient TTA prediction with NaN handling.

    Optimized for A100 with batch processing for faster inference.

    Args:
        models: List of trained models
        images: Test images in [0, 255] range, shape (N, H, W, 3)
        config: ExperimentConfig instance

    Returns:
        Mean predictions across all models and TTA augmentations
    """
    n_samples = len(images)
    n_models = len(models)
    n_tta = config.tta_augmentations if config.use_tta else 1

    # Initialize predictions accumulator
    all_predictions = np.zeros(n_samples, dtype=np.float32)
    prediction_count = 0

    # Batch size for TTA processing - use larger batches on A100
    tta_batch_size = min(config.batch_size_inference, 64)

    if not config.use_tta:
        # No TTA - simple batch prediction
        for model in models:
            for start_idx in range(0, n_samples, tta_batch_size):
                end_idx = min(start_idx + tta_batch_size, n_samples)
                batch = images[start_idx:end_idx].copy()

                # Apply EfficientNet preprocessing
                batch_preprocessed = efficientnet_preprocess(batch.astype(np.float32))

                # Predict
                preds = model.predict(batch_preprocessed, verbose=0, batch_size=tta_batch_size)
                preds_flat = preds.flatten()

                # Handle NaN - replace with 0.5 (uncertain)
                preds_flat = np.nan_to_num(preds_flat, nan=0.5)
                all_predictions[start_idx:end_idx] += preds_flat

                # Clean up
                del batch, batch_preprocessed, preds

            prediction_count += 1
            gc.collect()

        result = all_predictions / prediction_count
        result = np.nan_to_num(result, nan=0.5)
        return result

    # With TTA - process each image individually with augmentations
    print(f"Running TTA with {n_tta} augmentations on {n_samples} images ({n_models} models)...")

    nan_count = 0
    for img_idx in tqdm(range(n_samples), desc="TTA Prediction"):
        img = images[img_idx]
        img_predictions = []

        # Generate TTA augmentations for this single image
        tta_count = 0
        for aug_img in apply_tta_single(img):
            if tta_count >= n_tta:
                break

            # Prepare batch of 1
            aug_batch = np.expand_dims(aug_img, axis=0).astype(np.float32)
            aug_preprocessed = efficientnet_preprocess(aug_batch)

            # Predict with each model
            for model in models:
                pred = model.predict(aug_preprocessed, verbose=0, batch_size=1)
                pred_val = float(pred[0, 0])

                # Handle NaN predictions
                if np.isnan(pred_val) or np.isinf(pred_val):
                    nan_count += 1
                    pred_val = 0.5  # Default to uncertain

                img_predictions.append(pred_val)

            # Clean up
            del aug_batch, aug_preprocessed
            tta_count += 1

        # Average predictions for this image (use nanmean for safety)
        if img_predictions:
            mean_pred = np.nanmean(img_predictions)
            all_predictions[img_idx] = mean_pred if not np.isnan(mean_pred) else 0.5
        else:
            all_predictions[img_idx] = 0.5

        # Periodic garbage collection (less frequent on A100)
        if img_idx % 100 == 0:
            gc.collect()

    if nan_count > 0:
        print(f"Warning: {nan_count} NaN predictions replaced with 0.5")

    gc.collect()

    # Final safety check
    all_predictions = np.nan_to_num(all_predictions, nan=0.5)
    return all_predictions


def predict_with_tta(models, images, config):
    """
    Wrapper that calls memory-efficient TTA implementation.
    Maintains backward compatibility with existing code.
    """
    return predict_with_tta_memory_efficient(models, images, config)

## 12. Evaluation (V12 Pro)

**Comprehensive Evaluation:**
- Standard metrics at threshold=0.5
- Optimal threshold via Youden's J statistic
- Per-model individual results
- Ensemble voting analysis

In [ ]:
def evaluate_individual_models(models, test_images, test_labels, config):
    """
    Evaluate each model individually before ensemble.
    
    Returns per-model AUC for analysis.
    """
    print("\n" + "=" * 65)
    print("📊 INDIVIDUAL MODEL EVALUATION")
    print("=" * 65)
    
    individual_results = []
    
    for idx, model in enumerate(models):
        # Simple prediction without TTA for speed
        predictions = []
        batch_size = config.batch_size_inference
        
        for i in range(0, len(test_images), batch_size):
            batch = test_images[i:i+batch_size].copy()
            batch_preprocessed = efficientnet_preprocess(batch.astype(np.float32))
            preds = model.predict(batch_preprocessed, verbose=0)
            predictions.extend(preds.flatten())
        
        predictions = np.array(predictions)
        predictions = np.nan_to_num(predictions, nan=0.5)
        
        auc = roc_auc_score(test_labels, predictions)
        acc = accuracy_score(test_labels, (predictions > 0.5).astype(int))
        
        individual_results.append({
            'model_idx': idx,
            'auc': auc,
            'accuracy': acc,
            'predictions': predictions
        })
        
        print(f"   Model {idx + 1}: AUC={auc:.4f}, Acc={acc:.4f}")
    
    print("=" * 65)
    return individual_results


def evaluate_ensemble_v12pro(models, test_images, test_labels, config):
    """
    Comprehensive V12 Pro ensemble evaluation.
    
    Features:
    - Individual model evaluation
    - Ensemble with TTA
    - Optimal threshold calibration
    - Detailed metrics breakdown
    """
    # First evaluate individual models
    individual_results = evaluate_individual_models(models, test_images, test_labels, config)
    
    # Ensemble prediction with TTA
    print("\n" + "=" * 65)
    print("🎯 ENSEMBLE EVALUATION WITH TTA")
    print("=" * 65)
    
    predictions = predict_with_tta(models, test_images, config)
    
    # Ensure no NaN values
    predictions = np.nan_to_num(predictions, nan=0.5, posinf=1.0, neginf=0.0)
    predictions = np.clip(predictions, 0.0, 1.0)
    
    # Standard metrics at 0.5 threshold
    predicted_classes = (predictions > 0.5).astype(int)
    
    auc_score = roc_auc_score(test_labels, predictions)
    accuracy = accuracy_score(test_labels, predicted_classes)
    precision = precision_score(test_labels, predicted_classes, zero_division=0)
    recall = recall_score(test_labels, predicted_classes, zero_division=0)
    f1 = f1_score(test_labels, predicted_classes, zero_division=0)
    
    tn, fp, fn, tp = confusion_matrix(test_labels, predicted_classes).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Find optimal threshold using Youden's J statistic
    fpr, tpr, thresholds = roc_curve(test_labels, predictions)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    
    # Metrics at optimal threshold
    optimal_predicted = (predictions > optimal_threshold).astype(int)
    optimal_accuracy = accuracy_score(test_labels, optimal_predicted)
    optimal_precision = precision_score(test_labels, optimal_predicted, zero_division=0)
    optimal_recall = recall_score(test_labels, optimal_predicted, zero_division=0)
    optimal_f1 = f1_score(test_labels, optimal_predicted, zero_division=0)
    
    tn_opt, fp_opt, fn_opt, tp_opt = confusion_matrix(test_labels, optimal_predicted).ravel()
    optimal_specificity = tn_opt / (tn_opt + fp_opt) if (tn_opt + fp_opt) > 0 else 0
    
    # Calculate sensitivity at high specificity (clinical relevance)
    # Find threshold for 80% specificity
    spec_80_idx = np.argmin(np.abs((1 - fpr) - 0.80))
    sens_at_spec_80 = tpr[spec_80_idx] if spec_80_idx < len(tpr) else 0
    
    # Compile results
    results = {
        'version': 'V12_Pro',
        'ensemble_size': len(models),
        'tta_augmentations': config.tta_augmentations,
        
        # Standard threshold (0.5)
        'auc': float(auc_score),
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'sensitivity': float(recall),
        'specificity': float(specificity),
        'f1_score': float(f1),
        
        # Optimal threshold
        'optimal_threshold': float(optimal_threshold),
        'optimal_accuracy': float(optimal_accuracy),
        'optimal_precision': float(optimal_precision),
        'optimal_recall': float(optimal_recall),
        'optimal_sensitivity': float(optimal_recall),
        'optimal_specificity': float(optimal_specificity),
        'optimal_f1': float(optimal_f1),
        
        # Clinical relevance
        'sensitivity_at_80_specificity': float(sens_at_spec_80),
        
        # Confusion matrices
        'confusion_matrix': {
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
        },
        'optimal_confusion_matrix': {
            'tn': int(tn_opt), 'fp': int(fp_opt), 'fn': int(fn_opt), 'tp': int(tp_opt)
        },
        
        # Individual model results
        'individual_aucs': [r['auc'] for r in individual_results],
        
        # For plotting
        'predictions': predictions.tolist(),
        'true_labels': test_labels.tolist(),
        'fpr': fpr.tolist(),
        'tpr': tpr.tolist()
    }
    
    return results


# Run evaluation
results = evaluate_ensemble_v12pro(ensemble_models, test_images, test_labels, config)

# Print results
print("\n" + "=" * 65)
print("📈 V12 PRO FINAL RESULTS")
print("=" * 65)
print()
print("ENSEMBLE RESULTS (threshold=0.5):")
print(f"   AUC:           {results['auc']:.4f}")
print(f"   Accuracy:      {results['accuracy']:.4f}")
print(f"   Sensitivity:   {results['sensitivity']:.4f}")
print(f"   Specificity:   {results['specificity']:.4f}")
print(f"   F1 Score:      {results['f1_score']:.4f}")
print()
print(f"OPTIMAL THRESHOLD ({results['optimal_threshold']:.3f}):")
print(f"   Accuracy:      {results['optimal_accuracy']:.4f}")
print(f"   Sensitivity:   {results['optimal_sensitivity']:.4f}")
print(f"   Specificity:   {results['optimal_specificity']:.4f}")
print(f"   F1 Score:      {results['optimal_f1']:.4f}")
print()
print("CLINICAL METRIC:")
print(f"   Sensitivity @ 80% Specificity: {results['sensitivity_at_80_specificity']:.4f}")
print()
print("CONFUSION MATRIX (threshold=0.5):")
cm = results['confusion_matrix']
print(f"   TN={cm['tn']}, FP={cm['fp']}")
print(f"   FN={cm['fn']}, TP={cm['tp']}")
print()
print("INDIVIDUAL MODEL AUCS:")
for idx, auc in enumerate(results['individual_aucs']):
    print(f"   Model {idx + 1}: {auc:.4f}")
print()
print("=" * 65)

## 13. Visualization (V12 Pro)

**Diagnostic Plots:**
- ROC curve with AUC
- Training history with stage markers
- Confusion matrices at both thresholds
- Per-model comparison

In [ ]:
def plot_roc_curve_v12pro(results, save_path=None):
    """Plot ROC curve with AUC and optimal threshold marker."""
    plt.figure(figsize=(9, 9))
    
    # ROC curve
    plt.plot(results['fpr'], results['tpr'],
             color='#2196F3', lw=2.5,
             label=f"V12 Pro Ensemble (AUC = {results['auc']:.4f})")
    
    # Diagonal
    plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random')
    
    # Mark optimal threshold point
    opt_idx = np.argmax(np.array(results['tpr']) - np.array(results['fpr']))
    plt.scatter([results['fpr'][opt_idx]], [results['tpr'][opt_idx]], 
                c='red', s=100, zorder=5, 
                label=f"Optimal (thresh={results['optimal_threshold']:.3f})")
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
    plt.title('ROC Curve - EfficientNet-B4 V12 Pro', fontsize=14)
    plt.legend(loc='lower right', fontsize=11)
    plt.grid(True, alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrices_v12pro(results, save_path=None):
    """Plot confusion matrices at both thresholds."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Standard threshold
    cm_std = np.array([
        [results['confusion_matrix']['tn'], results['confusion_matrix']['fp']],
        [results['confusion_matrix']['fn'], results['confusion_matrix']['tp']]
    ])
    
    sns.heatmap(cm_std, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Benign', 'Malignant'],
                yticklabels=['Benign', 'Malignant'],
                annot_kws={'size': 14})
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('Actual', fontsize=11)
    axes[0].set_title(f"Threshold = 0.5\nAcc={results['accuracy']:.3f}, Sens={results['sensitivity']:.3f}, Spec={results['specificity']:.3f}", 
                      fontsize=12)
    
    # Optimal threshold
    cm_opt = np.array([
        [results['optimal_confusion_matrix']['tn'], results['optimal_confusion_matrix']['fp']],
        [results['optimal_confusion_matrix']['fn'], results['optimal_confusion_matrix']['tp']]
    ])
    
    sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', ax=axes[1],
                xticklabels=['Benign', 'Malignant'],
                yticklabels=['Benign', 'Malignant'],
                annot_kws={'size': 14})
    axes[1].set_xlabel('Predicted', fontsize=11)
    axes[1].set_ylabel('Actual', fontsize=11)
    axes[1].set_title(f"Threshold = {results['optimal_threshold']:.3f}\nAcc={results['optimal_accuracy']:.3f}, Sens={results['optimal_sensitivity']:.3f}, Spec={results['optimal_specificity']:.3f}", 
                      fontsize=12)
    
    plt.suptitle('V12 Pro - Confusion Matrices', fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_training_history_v12pro(histories, config, save_path=None):
    """Plot training history with stage markers."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    colors = ['#2196F3', '#4CAF50', '#FF9800']  # Blue, Green, Orange
    
    for idx, history in enumerate(histories):
        color = colors[idx % len(colors)]
        label_suffix = f" (Model {idx + 1})"
        
        epochs = range(len(history['loss']))
        
        # Loss
        axes[0, 0].plot(epochs, history['loss'], label=f'Train{label_suffix}', 
                        color=color, alpha=0.7)
        axes[0, 0].plot(epochs, history['val_loss'], label=f'Val{label_suffix}', 
                        color=color, linestyle='--', alpha=0.7)
        
        # AUC
        axes[0, 1].plot(epochs, history['auc'], label=f'Train{label_suffix}', 
                        color=color, alpha=0.7)
        axes[0, 1].plot(epochs, history['val_auc'], label=f'Val{label_suffix}', 
                        color=color, linestyle='--', alpha=0.7)
        
        # Accuracy
        axes[1, 0].plot(epochs, history['accuracy'], label=f'Train{label_suffix}', 
                        color=color, alpha=0.7)
        axes[1, 0].plot(epochs, history['val_accuracy'], label=f'Val{label_suffix}', 
                        color=color, linestyle='--', alpha=0.7)
        
        # Learning Rate
        if 'lr' in history:
            axes[1, 1].plot(epochs, history['lr'], label=label_suffix, 
                            color=color, alpha=0.7)
    
    # Add stage markers
    stage1_end = config.stage1_epochs
    stage2_end = config.stage1_epochs + config.stage2_epochs
    
    for ax in axes.flat:
        ax.axvline(x=stage1_end, color='red', linestyle=':', alpha=0.5, label='Stage 2 Start' if ax == axes[0,0] else '')
        ax.axvline(x=stage2_end, color='purple', linestyle=':', alpha=0.5, label='Stage 3 Start' if ax == axes[0,0] else '')
    
    # Styling
    axes[0, 0].set_title('Loss', fontsize=12)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend(fontsize=8, loc='upper right')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_title('AUC', fontsize=12)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('AUC')
    axes[0, 1].legend(fontsize=8, loc='lower right')
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_title('Accuracy', fontsize=12)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_title('Learning Rate', fontsize=12)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle('V12 Pro - Training History (Stage boundaries marked)', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Generate all plots
print("\n📊 Generating visualization plots...")

plot_roc_curve_v12pro(results, save_path=config.results_path / 'v12pro_roc_curve.png')
plot_confusion_matrices_v12pro(results, save_path=config.results_path / 'v12pro_confusion_matrices.png')
plot_training_history_v12pro(training_histories, config, save_path=config.results_path / 'v12pro_training_history.png')

print("✅ Visualizations saved to:", config.results_path)

## 14. Save Results (V12 Pro)

Save models, results, and training histories with comprehensive metadata.

In [ ]:
# ============================================================
# SAVE V12 PRO RESULTS
# ============================================================

print("\n" + "=" * 65)
print("💾 SAVING V12 PRO RESULTS")
print("=" * 65)

# Save each model
for idx, model in enumerate(ensemble_models):
    model_path = config.results_path / f'v12pro_model_{idx}.keras'
    model.save(str(model_path))
    print(f"   Model {idx + 1} saved: {model_path.name}")

# Compile comprehensive results
results_with_config = {
    'experiment': 'EfficientNetB4_V12_Pro',
    'version': 'V12_Pro',
    'timestamp': datetime.now().isoformat(),
    
    'architecture': {
        'base_model': 'EfficientNet-B4',
        'input_size': list(config.image_size),
        'dense_units': [config.dense_units, max(config.dense_units // 4, 128)],
        'dropout_rate': config.dropout_rate,
        'spatial_dropout_rate': config.spatial_dropout_rate,
        'l2_regularization': config.l2_regularization
    },
    
    'preprocessing': {
        'clahe_clip_limit': config.clahe_clip_limit,
        'bilateral_filter': config.use_bilateral_filter,
        'image_range': '[0, 255] -> [-1, 1] via efficientnet.preprocess_input'
    },
    
    'training': {
        'batch_size': config.batch_size,
        'ensemble_size': config.ensemble_size,
        'stage1_epochs': config.stage1_epochs,
        'stage1_lr': config.stage1_lr,
        'stage2_epochs': config.stage2_epochs,
        'stage2_lr': config.stage2_lr,
        'stage2_frozen_layers': config.stage2_frozen_layers,
        'stage3_epochs': config.stage3_epochs,
        'stage3_lr': config.stage3_lr,
        'stage3_frozen_layers': config.stage3_frozen_layers,
        'focal_loss': config.use_focal_loss,
        'focal_gamma': config.focal_gamma,
        'focal_alpha': config.focal_alpha,
        'cosine_annealing': config.use_cosine_annealing,
        'tta_augmentations': config.tta_augmentations
    },
    
    'data': {
        'train_samples': len(train_labels),
        'val_samples': len(val_labels),
        'test_samples': len(test_labels),
        'train_malignant': int((train_labels == 1).sum()),
        'test_malignant': int((test_labels == 1).sum())
    },
    
    'results': {
        'auc': results['auc'],
        'accuracy': results['accuracy'],
        'sensitivity': results['sensitivity'],
        'specificity': results['specificity'],
        'precision': results['precision'],
        'f1_score': results['f1_score'],
        'optimal_threshold': results['optimal_threshold'],
        'optimal_accuracy': results['optimal_accuracy'],
        'optimal_sensitivity': results['optimal_sensitivity'],
        'optimal_specificity': results['optimal_specificity'],
        'optimal_f1': results['optimal_f1'],
        'sensitivity_at_80_specificity': results['sensitivity_at_80_specificity'],
        'individual_aucs': results['individual_aucs']
    },
    
    'confusion_matrix': results['confusion_matrix'],
    'optimal_confusion_matrix': results['optimal_confusion_matrix'],
    
    'best_checkpoints': [t.summary() for t in best_trackers]
}

# Save main results
results_path = config.results_path / 'v12pro_results.json'
with open(results_path, 'w') as f:
    json.dump(results_with_config, f, indent=2)
print(f"   Results saved: {results_path.name}")

# Save training histories
for idx, history in enumerate(training_histories):
    history_path = config.results_path / f'v12pro_history_{idx}.json'
    serializable_history = {k: [float(v) for v in vals] for k, vals in history.items()}
    with open(history_path, 'w') as f:
        json.dump(serializable_history, f, indent=2)
    print(f"   History {idx + 1} saved: {history_path.name}")

# Save predictions for further analysis
predictions_path = config.results_path / 'v12pro_predictions.npz'
np.savez(
    predictions_path,
    predictions=np.array(results['predictions']),
    true_labels=np.array(results['true_labels']),
    fpr=np.array(results['fpr']),
    tpr=np.array(results['tpr'])
)
print(f"   Predictions saved: {predictions_path.name}")

print("=" * 65)
print(f"✅ All results saved to: {config.results_path}")
print("=" * 65)

## 15. Experiment Summary

Final summary comparing V12 Pro results with previous versions.

In [ ]:
print("=" * 70)
print("🏆 V12 PRO EXPERIMENT SUMMARY")
print("=" * 70)
print()
print("DATA CONFIGURATION:")
print(f"   Source:             CBIS-DDSM ROI Cropped Images")
print(f"   Image Size:         {config.image_size[0]}x{config.image_size[1]}")
print(f"   Training Samples:   {len(train_labels)}")
print(f"   Test Samples:       {len(test_labels)}")
print()
print("PREPROCESSING (ENHANCED):")
print(f"   CLAHE:              clip={config.clahe_clip_limit}")
print(f"   Bilateral Filter:   {config.use_bilateral_filter}")
print(f"   Normalization:      EfficientNet [-1, 1]")
print()
print("MODEL ARCHITECTURE:")
print(f"   Architecture:       EfficientNet-B4 V12 Pro")
print(f"   Dense Layers:       {config.dense_units} → {max(config.dense_units // 4, 128)}")
print(f"   Dropout:            {config.dropout_rate} + Spatial {config.spatial_dropout_rate}")
print(f"   Ensemble Size:      {config.ensemble_size}")
print()
print("TRAINING CONFIGURATION:")
print(f"   Stage 1 (Head):     {config.stage1_epochs} epochs @ LR={config.stage1_lr}")
print(f"   Stage 2 (Focus):    {config.stage2_epochs} epochs @ LR={config.stage2_lr}")
print(f"   Stage 3 (Fine):     {config.stage3_epochs} epochs @ LR={config.stage3_lr}")
print(f"   Best Selection:     Across ALL stages")
print(f"   TTA Augmentations:  {config.tta_augmentations}")
print()
print("─" * 70)
print("FINAL RESULTS:")
print("─" * 70)
print()
print(f"   AUC:                {results['auc']:.4f}")
print(f"   Accuracy:           {results['accuracy']:.4f}")
print(f"   Sensitivity:        {results['sensitivity']:.4f}")
print(f"   Specificity:        {results['specificity']:.4f}")
print(f"   F1 Score:           {results['f1_score']:.4f}")
print()
print(f"   Optimal Threshold:  {results['optimal_threshold']:.3f}")
print(f"   Optimal Sens:       {results['optimal_sensitivity']:.4f}")
print(f"   Optimal Spec:       {results['optimal_specificity']:.4f}")
print()
print("─" * 70)
print("COMPARISON WITH PREVIOUS VERSIONS:")
print("─" * 70)
print()
print("   Version       | Best Individual | Ensemble AUC | Notes")
print("   " + "-" * 60)
print(f"   V12 Original  | 0.7955          | 0.770        | Stage 2 best")
print(f"   V13 Optimized | 0.7870          | 0.770        | Stage 2 best")
print(f"   V12 Pro       | {max(results['individual_aucs']):.4f}          | {results['auc']:.4f}        | Best across stages")
print()

# Check if we improved
best_individual = max(results['individual_aucs'])
improvement_individual = best_individual - 0.7955
improvement_ensemble = results['auc'] - 0.770

if improvement_ensemble > 0:
    print(f"   ✅ Ensemble AUC improved by {improvement_ensemble:.4f}")
else:
    print(f"   📊 Ensemble AUC delta: {improvement_ensemble:+.4f}")

if improvement_individual > 0:
    print(f"   ✅ Best individual improved by {improvement_individual:.4f}")
else:
    print(f"   📊 Best individual delta: {improvement_individual:+.4f}")

print()
print("=" * 70)
print("BEST CHECKPOINTS:")
print("=" * 70)
for idx, tracker in enumerate(best_trackers):
    print(f"   Model {idx + 1}: {tracker.summary()}")
print()
print("=" * 70)
print(f"📁 Results saved to: {config.results_path}")
print("=" * 70)